# Scrape full job details for each job

For each job (you can go by URL), please try to get the following pieces of information:
1. Job Description
2. Do they ask additional questions?
3. Company Information (May not exist for some firms)
• Company name

• Industry

• Firm size

• Company description

• Perks and benefits

• Average rating

• Number of reviews

• Any other information

Please get this information for the most recent 1000 jobs and report how long the
scraping takes

# Selenium

In [1]:
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver

# Set up the driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

def get_job_urls_from_page(page_num):
    URL = f"https://sg.jobstreet.com/jobs?page={page_num}&sortmode=ListedDate"
    driver.get(URL)
    time.sleep(3)  # Wait for the page to load

    job_urls = []
    job_cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="job-card"]')
    
    for job in job_cards:
        url_tag = job.find_element(By.CSS_SELECTOR, '[data-automation="job-list-view-job-link"]')
        job_url = url_tag.get_attribute('href') if url_tag else 'N/A'
        job_urls.append(job_url)
    
    return job_urls

# Get job URLs from multiple pages
job_urls = []
page_num = 1
while len(job_urls) < 20:
    job_urls.extend(get_job_urls_from_page(page_num))
    page_num += 1
    if len(job_urls) >= 20:
        break

# Remove duplicates (if any)
job_urls = list(set(job_urls))[:20]


In [2]:
len(job_urls)

20

In [3]:
def get_job_details(job_url):
    driver.get(job_url)
    time.sleep(3)  # Wait for the job page to load

    job_details = {}
    
    # Job Description
    try:
        description_tag = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-automation="jobAdDetails"]'))
        )
        job_details['Job Description'] = description_tag.text.strip() if description_tag else 'N/A'
    except Exception as e:
        job_details['Job Description'] = 'N/A'

    # Employer Questions
    try:
        questions_section = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, "//h2[contains(text(), 'Employer questions')]/following-sibling::div//ul"))
        )
        question_items = questions_section.find_elements(By.XPATH, ".//li")
        questions = [item.text.strip() for item in question_items if item.text.strip()]
        job_details['Employer Questions'] = questions if questions else ['N/A']
    except Exception as e:
        job_details['Employer Questions'] = ['N/A']
    
    return job_details

# Extract details for each job URL
job_data = []
for job_url in job_urls:
    try:
        job_details = get_job_details(job_url)
        job_data.append(job_details)
    except Exception as e:
        print(f"Error occurred while processing {job_url}: {e}")
        job_data.append({'Job Description': 'N/A', 'Employer Questions': 'N/A'})

# Create DataFrame
df = pd.DataFrame(job_data)

# Save to Excel
df.to_excel("20_job_details_with_questions.xlsx", index=False)
print("Job details saved to Excel file.")


Job details saved to Excel file.


In [4]:
driver.quit()

37m 10.2s for 200 jobs with job description and additional questions only
3m 26s for 20 jobs with job description and additional questions only